## PDF Parsing with Tamil OCR

This notebook extracts text from PDF files using OCR with Tamil and English language support. 

### Features
- **Single PDF Processing**: Extract text from individual PDF files
- **Batch Processing**: Process multiple PDFs from folder structures while preserving directory hierarchy
- **Tamil + English OCR**: Uses the `ocr_tamil` model for accurate text extraction in both languages
- **Automatic Formatting**: Extracted text is formatted into clean lines (6 words per line) for readability

### Setup
1. Ensure all dependencies are installed (PyMuPDF, ocr_tamil, torch, timm)
2. The notebook includes a compatibility patch for `timm>=1.0.0` with `ocr_tamil`
3. Model weights are loaded from `../model_weights/` directory

### Usage
- **Single PDF**: Call `parse_single_pdf(pdf_path, output_file_path)`
- **Batch Processing**: Call `process_pdf_folder(input_folder, output_folder)` with zone folders (CZ, SZ, WZ, NZ)

In [1]:
import os
import fitz  # PyMuPDF
from ocr_tamil.ocr import OCR

# Patch for timm Attention compatibility
import timm
from timm.layers.attention import Attention as OriginalAttention
import torch.nn as nn


def patched_forward(self, x, attn_mask=None):
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(
        2, 0, 3, 1, 4
    )
    q, k, v = qkv[0], qkv[1], qkv[2]
    attn = (q @ k.transpose(-2, -1)) * self.scale
    if attn_mask is not None:
        attn = attn + attn_mask
    attn = attn.softmax(dim=-1)
    attn = self.attn_drop(attn)
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x


OriginalAttention.forward = patched_forward

ocr_detect_tamil = OCR(
    detect=True,
    lang=["tamil", "english"],
    batch_size=128,
    tamil_model_path=r"../model_weights/parseq_tamil_v3.pt",
    eng_model_path=r"../model_weights/craft_mlt_25k.pth",
)

import pathlib
project_root = pathlib.Path("/home/arunachalam/Projects/cbcid")


def parse_single_pdf(pdf_path: str, output_file_path: str) -> str:
    """Extracts text from a single PDF file using OCR and saves it to an output file."""
    import tempfile

    pdf_document = fitz.open(pdf_path)
    all_texts = []

    temp_dir = tempfile.gettempdir()

    for page_num in range(len(pdf_document)):
        page = pdf_document.load_page(page_num)
        pix = page.get_pixmap()

        temp_img = os.path.join(temp_dir, f"temp_page_{page_num}.png")
        pix.save(temp_img)

        try:
            texts = ocr_detect_tamil.predict(temp_img)
            full_text = " ".join(texts[0])
            words = full_text.split()
            formatted_text = "\n".join(
                " ".join(words[i : i + 6]) for i in range(0, len(words), 6)
            )
            all_texts.append(formatted_text)
        finally:
            if os.path.exists(temp_img):
                os.remove(temp_img)

    final_text = "\n\n".join(all_texts)

    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    return final_text


def process_pdf_folder(input_folder: str, output_folder: str):
    """Processes all PDFs in a folder and its subfolders."""
    # Resolve paths
    input_path = (
        pathlib.Path(input_folder)
        if input_folder.startswith("/")
        else project_root / input_folder
    )
    output_path = (
        pathlib.Path(output_folder)
        if output_folder.startswith("/")
        else project_root / output_folder
    )

    if not input_path.exists():
        print(f"Warning: Input folder '{input_path}' does not exist.")
        return

    total_pdf_count = 0
    pdf_files = sorted(input_path.glob("**/*.pdf"))

    for pdf_file in pdf_files:
        relative_path = pdf_file.relative_to(input_path)

        location = "root" if relative_path.parent == pathlib.Path(".") else str(
            relative_path.parent
        )

        print(f"Processing: {pdf_file.name} in {location}")

        output_dir = output_path / relative_path.parent
        output_dir.mkdir(parents=True, exist_ok=True)

        # Critical: compute a fresh output_file_path for each file
        output_file_path = output_dir / f"{pdf_file.stem}_parsed.txt"

        parse_single_pdf(str(pdf_file), str(output_file_path))

        clean_path = str(output_file_path).split("data/")[-1]
        print(f"Saved → data/{clean_path}\n")

        total_pdf_count += 1

    if total_pdf_count == 0:
        print(f"No PDFs found in {input_path}")
    else:
        print(f"Successfully processed {total_pdf_count} PDFs!")

In [2]:
input_folder = "data/FIRs_input_folder/SZ"
output_folder = "data/Parsed_FIRs_output_folder"

process_pdf_folder(input_folder, output_folder)

Processing: ATC, Cr. No. 04 of 2017.pdf in root
Saved → data/Parsed_FIRs_output_folder/ATC, Cr. No. 04 of 2017_parsed.txt

Processing: CBCID ATC Cr. No. 01of 2014.pdf in root
Saved → data/Parsed_FIRs_output_folder/CBCID ATC Cr. No. 01of 2014_parsed.txt

Successfully processed 2 PDFs!
